---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📝 **Student**: Leonard Chiang (repo [`@lchiangnyc`](https://github.com/lchiangnyc/ai-engineering-fordham))

### 📋 **Homework 4**: Embeddings & Semantic Search

### 📅 **Due Date**: Day of Lecture 5, 11:59 PM


**Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

In this homework, you'll build on Homework 3 (BM25 search) by adding **embedding-based semantic search**.

You will:
1. **Generate embeddings** using both local (Hugging Face) and API (OpenAI) models
2. **Implement cosine similarity** from scratch
3. **Implement semantic search** from scratch
4. **Compare BM25 vs semantic search** using Recall
5. **Compare different embedding models** and analyze their differences

**Total Points: 95**

---

## Instructions

- Complete all tasks by filling in code where you see `# YOUR CODE HERE`
- You may use ChatGPT, Claude, documentation, Stack Overflow, etc.
- When using external resources, briefly cite them in a comment
- Run all cells before submitting to ensure they work

**Submission:**
1. Create a branch called `homework-4`
2. Commit and push your work
3. Create a PR and merge to main
4. Submit the `.ipynb` file on Blackboard

---

## Task 1: Environment Setup (10 points)

### 1a. Imports (5 pts)

Import the required libraries and load the WANDS data.

In [1]:
# ruff: noqa: E402

# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings("ignore")
print(f"Standard imports successful!")

# Import ONLY data loading from helpers
import sys
sys.path.append('../scripts')
from helpers import load_wands_products, load_wands_queries, load_wands_labels
print(f"helpers imports successful!")

# Embedding libraries - we use these directly
from sentence_transformers import SentenceTransformer
import litellm
print(f"Embedding imports successful!")

# Load environment variables for API keys
from dotenv import load_dotenv
load_dotenv()
print(f"Environment imports successful!")

pd.set_option('display.max_colwidth', 80)
print("All prescribed imports successful!")

Standard imports successful!
helpers imports successful!
Embedding imports successful!
Environment imports successful!
All prescribed imports successful!


In [2]:
# Drawing on Homework 3
from collections import Counter
import string
from pathlib import Path
# a stemmer from `pystemmer` for better tokenization
import Stemmer 

# Bringing in progress bar
    # Ref. tqdm documentation https://tqdm.github.io/docs/tqdm/
from tqdm import tqdm
from tqdm.gui import tqdm as tqdm_gui

# Accessing environmental variables
import os

# Examining tokenization
import tiktoken

print(f"Leonard's additional imports successful!")

Leonard's additional imports successful!


In [3]:
# Load the WANDS dataset
dir_path = "C:/Users/leona/ai-engineering-fordham/data"     # dir for directory

products = load_wands_products(dir_path)
queries = load_wands_queries(dir_path)
labels = load_wands_labels(dir_path)

print(f"Products: {len(products):,}")
print(f"Queries: {len(queries):,}")
print(f"Labels: {len(labels):,}")

Products: 42,994
Queries: 480
Labels: 233,448


### 1b. Copy BM25 functions from HW3 (5 pts)

Copy your BM25 implementation from Homework 3. We'll use it to compare against semantic search.

In [4]:
# Copy your BM25 functions from Homework 3

# Provided functions - run this cell to define them

stemmer = Stemmer.Stemmer('english')
punct_trans = str.maketrans({key: ' ' for key in string.punctuation})

def snowball_tokenize(text: str) -> list[str]:
    """
    Tokenize text with Snowball stemming.
    
    Args:
        text: The text to tokenize
        
    Returns:
        List of stemmed tokens
    """
    if pd.isna(text) or text is None:
        return []
    text = str(text).translate(punct_trans)
    tokens = text.lower().split()
    return [stemmer.stemWord(token) for token in tokens]

def build_index(docs: list[str], tokenizer) -> tuple[dict, list[int]]:
    """
    Build an inverted index from a list of documents.
    
    Args:
        docs: List of document strings to index
        tokenizer: Function that takes text and returns list of tokens
        
    Returns:
        index: dict mapping term -> {doc_id: term_count}
        doc_lengths: list of document lengths (in tokens)
    """
    index = {}
    doc_lengths = []
    
    for doc_id, doc in enumerate(docs):
        tokens = tokenizer(doc)
        doc_lengths.append(len(tokens))
        term_counts = Counter(tokens)
        
        for term, count in term_counts.items():
            if term not in index:
                index[term] = {}
            index[term][doc_id] = count
    
    return index, doc_lengths

def get_tf(term: str, doc_id: int, index: dict) -> int:
    """
    Get term frequency for a term in a document.
    
    Args:
        term: The term to look up
        doc_id: The document ID
        index: The inverted index
        
    Returns:
        Term frequency (count), or 0 if not found
    """
    if term in index and doc_id in index[term]:
        return index[term][doc_id]
    return 0

def get_df(term: str, index: dict) -> int:
    """
    Get document frequency for a term.
    
    Args:
        term: The term to look up
        index: The inverted index
        
    Returns:
        Number of documents containing the term
    """
    if term in index:
        return len(index[term])
    return 0

def bm25_idf(df: int, num_docs: int) -> float:
    """
    BM25 IDF formula.
    
    Args:
        df: Document frequency
        num_docs: Total number of documents
        
    Returns:
        IDF score
    """
    return np.log((num_docs - df + 0.5) / (df + 0.5) + 1)

def bm25_tf(tf: int, doc_len: int, avg_doc_len: float, k1: float = 1.2, b: float = 0.75) -> float:
    """
    BM25 TF normalization.
    
    Args:
        tf: Term frequency
        doc_len: Document length in tokens
        avg_doc_len: Average document length
        k1: Saturation parameter (default 1.2)
        b: Length normalization (default 0.75)
        
    Returns:
        Normalized TF score
    """
    return (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_len / avg_doc_len))

def score_bm25(query: str, index: dict, num_docs: int, doc_lengths: list[int], 
               tokenizer, k1: float = 1.2, b: float = 0.75) -> np.ndarray:
    """
    Score all documents using BM25.
    
    Args:
        query: The search query
        index: Inverted index
        num_docs: Total number of documents
        doc_lengths: List of document lengths
        tokenizer: Tokenization function
        
    Returns:
        Array of scores for each document
    """
    query_tokens = tokenizer(query)
    scores = np.zeros(num_docs)
    avg_doc_len = np.mean(doc_lengths) if doc_lengths else 1.0
    
    for token in query_tokens:
        df = get_df(token, index)
        if df == 0:
            continue
        
        idf = bm25_idf(df, num_docs)
        
        if token in index:
            for doc_id, tf in index[token].items():
                tf_norm = bm25_tf(tf, doc_lengths[doc_id], avg_doc_len, k1, b)
                scores[doc_id] += idf * tf_norm
    
    return scores

def search_products(query: str, products_df: pd.DataFrame, index: dict, 
                    doc_lengths: list[int], tokenizer, k: int = 10) -> pd.DataFrame:
    """
    Search products and return top-k results.
    
    Args:
        query: The search query
        products_df: DataFrame of products
        index: Inverted index
        doc_lengths: Document lengths
        tokenizer: Tokenization function
        k: Number of results to return
        
    Returns:
        DataFrame with top-k products and scores
    """
    scores = score_bm25(query, index, len(products_df), doc_lengths, tokenizer)
    top_k_idx = np.argsort(-scores)[:k]
    
    results = products_df.iloc[top_k_idx].copy()
    results['score'] = scores[top_k_idx]
    results['rank'] = range(1, k + 1)
    return results

print("All functions defined!")

All functions defined!


---

## Task 2: Understanding Embeddings (15 points)

### 2a. Load a local model and generate embeddings (5 pts)

Use `sentence-transformers` to load a local embedding model and generate embeddings for a list of words.

> _**<u>NOTE TO READER</u>:** I call on the_ `tqdm` _package to display progress bars for iterative operations. Relevant dependencies are listed in the_ `pyproject.toml` _and_ `uv.lock` _files... wherein_ `tqdm` _itself is listed as a dependency for other packages deployed in this course, so (given the probable audience of this notice) it's quite possible no additional installations are needed._

In [5]:
# Load the all-MiniLM-L6-v2 model using SentenceTransformer
# Then generate embeddings for each word in the list
words = ["wooden coffee table",
    "oak dining table",
    "red leather sofa",
    "blue area rug",
    "kitchen sink"
    ]

# YOUR CODE HERE
    # Ref. SentenceTransformer documentation https://sbert.net/

# Loading specified model
local_model = "all-MiniLM-L6-v2"
model = SentenceTransformer(local_model)

# Calculating embeddings through model.encode()
    # Ack. Prof. Matt Murphy's Quantitative Foundations course, Homework 1b
    #   for list-tqdm-vstack workflow
words_embeddings_aml6_list = []     # ml6 for all-MiniLM-L6-v2
for phrase in tqdm(words, desc = "Phrase embeds."):
    words_embeddings_aml6_list.append(
        model.encode(phrase)
        )

words_embeddings_aml6 = np.vstack(words_embeddings_aml6_list)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 208.80it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Phrase embeds.: 100%|██████████| 5/5 [00:00<00:00, 13.91it/s]


_Earlier I was asked only to import the data-loading functions from_ `helpers.py` _file, apparently to the exclusion of the_ `batch_embed_local()` _function that the Lecture 4 script utilizes; accordingly, I am using another (considerably slower?) implementation._

In [6]:
# Print the number of embeddings you generated and the dimension of the embeddings
# Using if control flow to validate vstack configuration
if (len(words) == words_embeddings_aml6.shape[0]):
    print(f"== EMBEDDING SHAPE: {local_model} ==",
        f"\nNumber of embeddings generated: {words_embeddings_aml6.shape[0]},",
        f"\n  each of {words_embeddings_aml6.shape[1]} dimensions")

== EMBEDDING SHAPE: all-MiniLM-L6-v2 == 
Number of embeddings generated: 5, 
  each of 384 dimensions


### 2b. Implement cosine similarity and create a similarity matrix (5 pts)

Implement cosine similarity from scratch:

$$\text{cosine\_similarity}(a, b) = \frac{a \cdot b}{\|a\| \times \|b\|}$$

In [7]:
# Implement cosine similarity from scratch
def cos_sim(a, b):     # cos_sim for cosine similarity
    # Using numpy library for economy of computing and langauge
    dot_prod_ab = np.dot(a, b)     # prod for product
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)

    cosine_similarity = dot_prod_ab / (norm_a * norm_b)

    return cosine_similarity

# Create similarity matrix
# Initializing as matrix of zeros
    # Ref. https://numpy.org/devdocs/reference/generated/numpy.zeros.html
words_aml6_sim_mat = np.zeros((5, 5))     # sim_mat for similarity matrix

# Modifying contents using index positions
for i in range(len(words_embeddings_aml6)):
    for j in range(len(words_embeddings_aml6)):
        words_aml6_sim_mat[i][j] = cos_sim(
            words_embeddings_aml6[i], words_embeddings_aml6[j]
            )

print(words_aml6_sim_mat)

[[1.         0.58863068 0.37062213 0.18948629 0.29571229]
 [0.58863068 1.         0.33790976 0.24952091 0.34140956]
 [0.37062213 0.33790976 1.         0.38031045 0.05773977]
 [0.18948629 0.24952091 0.38031045 1.         0.12580225]
 [0.29571229 0.34140956 0.05773977 0.12580225 1.        ]]


In [8]:
# Display as DataFrame
display(pd.DataFrame(words_aml6_sim_mat))

,0,1,2,3,4
0,1.000000,0.588631,0.370622,0.189486,0.295712
1,0.588631,1.000000,0.337910,0.249521,0.341410
2,0.370622,0.337910,1.000000,0.380310,0.057740
3,0.189486,0.249521,0.380310,1.000000,0.125802
4,0.295712,0.341410,0.057740,0.125802,1.000000


In [9]:
# Comparing to SentenceTransformer's built-in similarity calculator
    # Ref. SentenceTransformer documentation https://sbert.net/ again
words_sm_st = model.similarity(
    words_embeddings_aml6, words_embeddings_aml6)      # sm_st for similiarity matrix from SentenceTransformer
print(words_sm_st)

tensor([[1.0000, 0.5886, 0.3706, 0.1895, 0.2957],
        [0.5886, 1.0000, 0.3379, 0.2495, 0.3414],
        [0.3706, 0.3379, 1.0000, 0.3803, 0.0577],
        [0.1895, 0.2495, 0.3803, 1.0000, 0.1258],
        [0.2957, 0.3414, 0.0577, 0.1258, 1.0000]])


_They agree (to four decimal places)._

### 2c. Embed using OpenAI API (5 pts)

Use `litellm` to get embeddings from OpenAI's API and compare dimensions.

In [10]:
# Checking Notebook can draw key from .env
os.environ.get("OPENAI_API_KEY")
print(f"Length of OpenAI API key: {len(os.environ["OPENAI_API_KEY"])} characters")

Length of OpenAI API key: 164 characters


In [11]:
# Use litellm to get an embedding from OpenAI's text-embedding-3-small model
# Compare the dimension with the local model
api_model = "text-embedding-3-small"
words_response_te3s = litellm.embedding(
    model = api_model,
    input = words)

In [12]:
# Showing response in advance of data extraction
print(f"{words_response_te3s.model_dump_json(indent = 2)}")

{
  "model": "text-embedding-3-small",
  "data": [
    {
      "embedding": [
        -0.024980908,
        -0.010949807,
        0.027767058,
        -0.049914595,
        0.01914888,
        -0.050386824,
        0.015146742,
        0.034307428,
        0.0047606574,
        -0.0338352,
        0.025051743,
        -0.026232315,
        -0.0016587039,
        -0.017354412,
        -0.032087952,
        0.038510267,
        0.014296729,
        -0.01906624,
        -0.009544927,
        0.048828468,
        -0.014993267,
        0.048781242,
        -0.003777831,
        -0.010530704,
        0.05430632,
        0.0536452,
        0.034850493,
        0.010442161,
        -0.031119883,
        0.0023404844,
        -0.04398812,
        -0.022655182,
        -0.0018062755,
        -0.0109852245,
        0.014438398,
        -0.06474258,
        -0.01374186,
        -0.040186677,
        -0.035204664,
        -0.021816975,
        -0.009580343,
        0.026917046,
        0.03369353,


In [13]:
words_embeddings_te3s_list = []     # te3s for text-embedding-3-small
for i in range(len(words)):
    words_embeddings_te3s_list.append(
        list(words_response_te3s.data[i]["embedding"])
        )

words_embeddings_te3s = np.vstack(words_embeddings_te3s_list)

# Using if control flow to validate vstack configuration
api_model = "text-embedding-3-small"
if (len(words) == words_embeddings_te3s.shape[0]):
    print(f"== EMBEDDING SHAPE: {api_model} ==",     # DIMS. for dimensions
        f"\nNumber of embeddings generated: {words_embeddings_te3s.shape[0]},",
        f"\n  each of {words_embeddings_te3s.shape[1]} dimensions")

== EMBEDDING SHAPE: text-embedding-3-small == 
Number of embeddings generated: 5, 
  each of 1536 dimensions


---

## Task 3: Batch Embedding Products (20 points)

### 3a. Embed a product sample (10 pts)

Create a combined text field and embed 5,000 products using the local model.

_Cit. blogpost from [Michael Galarnyk on Medium](https://medium.com/data-science/understanding-sampling-with-and-without-replacement-python-7aff8f47ebe4) for decision to sample without replacement.<br><br>Relevant syntax [from pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html):_ `DataFrame.sample(n=None, frac=None, replace=False, weights=None, random_state=None, axis=None, ignore_index=False)`

| _Parameter_ | _Action_ | _Motivation_ |
| - | - | - |
| `n` | _Set_ `= 5000` | _Per instructions_ |
| `frac` | _Omit_ | _(Cannot be used alongside_ `n`_)_ |
| `replace` | _Keep as_ `False` | _Don't want repeats_ |
| `weights` | _Keep as_ `None` | _Treating all attributes equally_ |
| `random_state` | _Select and disclose_ | _Taking "consistent" to mean reproducible_ |
| `axis` | _Set_ `= "rows"` | _Sampling rows_ |
| `ignore_index` | _Keep as_ `False` | _No need to change_ |

_Drawing on Lecture 4 code, just retaining index to confirm random sampling,_

In [14]:
# Get a consistent sample
    # Ref. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html

# Randomly sampling
products_sample_DF = products.sample(
    n = 5000,
    replace = False,
    weights = None,
    random_state = 168,
    axis = "rows",
    ignore_index = False
    )
print(products_sample_DF.shape)

(5000, 9)


In [15]:
# Create a combined text field (product_name + product_class)
# Then embed all products using model.encode()

# YOUR CODE HERE
# Combining text fields by concatenation, replacing NaN with empty string
products_sample_DF["embed_text"] = (
    products_sample_DF["product_name"].fillna("") + 
    " " + products_sample_DF["product_class"].fillna("")
    )

display(products_sample_DF["embed_text"].head())

14954    orahh 28.5 '' wide outdoor patio sofa with sunbrella cushions Patio Sofas
18511     rack shelves for server racks cabinet Rackmounts & Rackmount Accessories
19421                               infinity solid wood dining chair Dining Chairs
22861                                        cublington outdoor 7 piece patio set 
17239                door mats go away 30 in . x 18 in . outdoor door mat Doormats
Name: embed_text, dtype: str

**_OBSERVATIONS<br>_**

| _What I see_ | _Reaction_ |
| - | - |
| _Scrambled row indices_ | _Encouraging evidence of random sampling success_ |
| _Loose punctuation, e.g. "_`''`_" (index 14954)_ | _Consider stripping punctuation,<br>to save tokens on high-volume job_ |

In [16]:
product_embeddings_list = []
for text in tqdm(products_sample_DF["embed_text"], desc = "Local embeds."):
    product_embeddings_list.append(model.encode(text))

product_embeddings = np.vstack(product_embeddings_list)

Local embeds.: 100%|██████████| 5000/5000 [38:57<00:00,  2.14it/s]    


In [17]:
product_embeddings.shape

(5000, 384)

_Looks good!_

### 3b. Save and load embeddings (5 pts)

Save embeddings to a `.npy` file so you don't have to recompute them.

In [18]:
# Save embeddings to ../temp/hw4_embeddings.npy
np.save("temp/hw4_embeddings.npy", product_embeddings)

# Save products_sample to ../temp/hw4_products.csv
products_sample_DF.to_csv("temp/hw4_products.csv", index = False)

In [19]:
np.load("C:/Users/leona/ai-engineering-fordham/temp/hw4_embeddings.npy")

array([[ 0.08509231,  0.01606525,  0.00730968, ..., -0.00536601,
         0.0111773 ,  0.02960096],
       [ 0.01781675,  0.00493398, -0.05941138, ..., -0.06560021,
        -0.09401219, -0.10542288],
       [ 0.00204732, -0.0064519 , -0.04262346, ...,  0.02001614,
        -0.03582562,  0.003653  ],
       ...,
       [ 0.02741452,  0.01026168, -0.03989461, ...,  0.04437125,
         0.05642774, -0.00774798],
       [ 0.08314803,  0.02412694, -0.0241473 , ..., -0.08078106,
         0.04714797, -0.05168823],
       [ 0.00276813,  0.07500771, -0.00628389, ...,  0.07104445,
         0.00505084, -0.05251857]], shape=(5000, 384), dtype=float32)

In [20]:
pd.read_csv("C:/Users/leona/ai-engineering-fordham/temp/hw4_products.csv")

,product_id,product_name,product_class,category_hierarchy,product_description,product_features,rating_count,average_rating,review_count,embed_text
0,14954,orahh 28.5 '' wide outdoor patio sofa with sunbrella cushions,Patio Sofas,Outdoor / Outdoor & Patio Furniture / Outdoor Seating & Patio Chairs / Patio...,"this product is the perfect addition to any patio , deck , or outdoor area ....",seatheightwithoutcushion:15|piecesincluded:1 armless chair and 1 ottoman|cus...,23.0,5.0,18.0,orahh 28.5 '' wide outdoor patio sofa with sunbrella cushions Patio Sofas
1,18511,rack shelves for server racks cabinet,Rackmounts & Rackmount Accessories,"School Furniture and Supplies / School Boards & Technology / AV, Mounts & Te...",the universally compatible rack shelf is perfect for holding all equipment r...,overalldepth-fronttoback:12.5|weightcapacity:45|mountingtype : cage nut|fini...,NaN,NaN,NaN,rack shelves for server racks cabinet Rackmounts & Rackmount Accessories
2,19421,infinity solid wood dining chair,Dining Chairs,Furniture / Kitchen & Dining Furniture / Dining Tables & Seating / Kitchen &...,infinity is a industrial style chair available with upholstered seat . the l...,overallheight-toptobottom:32.2|seatheight-floortoseat:17.3|warrantylength:3 ...,3.0,3.5,3.0,infinity solid wood dining chair Dining Chairs
3,22861,cublington outdoor 7 piece patio set,NaN,Outdoor / Outdoor & Patio Furniture / Patio Furniture Sets / Patio Dining Sets,outdoor practicality meets dining room style with this wonderful extendable ...,numberofchairsincluded:6|producttype : dining set|style : modern & contempor...,NaN,NaN,NaN,cublington outdoor 7 piece patio set
4,17239,door mats go away 30 in . x 18 in . outdoor door mat,Doormats,Rugs / Doormats,NaN,overallwidth-fronttoback:18|resistancetype : mildew resistant|warrantylength...,28.0,5.0,21.0,door mats go away 30 in . x 18 in . outdoor door mat Doormats
...,...,...,...,...,...,...,...,...,...,...
4995,40367,argyros california king bed,Protection Plans,Protection Plans,this bedroom set uses v-patterned wood panels to create an elegant display o...,lengthofwarranty:1 year,NaN,NaN,NaN,argyros california king bed Protection Plans
4996,18514,ernis power lift assist recliner,Recliners,Furniture / Living Room Furniture / Chairs & Seating / Recliners,"recliners chair for elderly : it reclines to 135 degrees , extending footres...",cushionconstruction : foam|warrantylength:1 year|productcare : clean with dr...,NaN,NaN,NaN,ernis power lift assist recliner Recliners
4997,6982,office chair,Office Chairs,Furniture / Office Furniture / Office Chairs,the office chair is ideal for any casual or professional working area . it p...,backupholsteryfillmaterial : foam|minimumoverallheight-toptobottom:31|warran...,5.0,4.0,4.0,office chair Office Chairs
4998,41765,kham bed frame,Bed Frames,NaN,"this product is crafted from wood , its low platform lends a casual look , w...","materialdetails : rubberwood , mdf wood , and lvl|overalllength-headtotoe:79...",7.0,3.0,6.0,kham bed frame Bed Frames


### 3c. Cost estimation (5 pts)

Estimate the cost to embed all 43K products using OpenAI's API.

**Pricing**: text-embedding-3-small costs ~$0.02 per 1 million tokens.

In [21]:
api_model

'text-embedding-3-small'

In [22]:
# Use tiktoken to count actual tokens in the sample
# Then extrapolate to estimate cost for the full dataset

# Loading an encoding
    # Ref. https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
encoding = tiktoken.encoding_for_model(api_model)
# encoding = tiktoken.get_encoding("cl100k_base")

_[tiktoken documentation indicates](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb) that the_ `text-embedding-3-small` _model uses_ `cl100k_base` _encoding._

In [23]:
# Adopting function defined in documentation
    # Adapting by setting appropriate default argument for encoding
def num_tokens_from_string(string: str,
    encoding_name: str = "cl100k_base"
    ) -> int:
    
    """Returns the number of tokens in a text string."""
    
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    
    return num_tokens

In [24]:
tokens_sample_ct = 0     # ct for count
for text in tqdm(products_sample_DF["embed_text"],
    desc = "Local embeds. token count"):
    tokens_sample_ct += num_tokens_from_string(text)

print(tokens_sample_ct)

Local embeds. token count: 100%|██████████| 5000/5000 [00:00<00:00, 18588.33it/s]

65721


_Time for some algebra._
$$ \text{tokens}_\text{ full dataset} = \text{tokens}_\text{ sample} \cdot \frac{\text{products}_\text{ full}}{\text{products}_\text{ sample}} \text{ ;}$$

$$ \text{cost}_\text{ full dataset} = \left( \text{tokens}_\text{ sample} \cdot \frac{\text{products}_\text{ full}}{\text{products}_\text{ sample}} \right) \cdot \frac{\$0.02}{1 \times 10^{6} \text{ tokens}} $$

In [25]:
products_full_ct, products_sample_ct = len(products), 5000     # ct for count
tokens_rate = (0.02) / (1 * (10 ** 6))

tokens_full_ct = tokens_sample_ct * ((products_full_ct) / (products_sample_ct)) 
cost_full = tokens_full_ct * tokens_rate
print(f"== COST ESTIMATE ==",
    f"\n${cost_full:.2f} for {int(tokens_full_ct) + 1} tokens")

== COST ESTIMATE == 
$0.01 for 565122 tokens


---

## Task 4: Semantic Search (25 points)

### 4a. Implement semantic search (15 pts)

Implement a semantic search function from scratch.

In [26]:
type(product_embeddings)

numpy.ndarray

In [27]:
# Implement batch cosine similarity for efficiency
    # Ref. https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html
def batch_cos_sim(query_embed, doc_embeds):
    query_unit = query_embed / np.linalg.norm(query_embed)     # unit for unit-length vector
    doc_units = doc_embeds / np.linalg.norm(
        doc_embeds, axis = 1, keepdims = True)
    
    return np.dot(doc_units, query_unit)

In [28]:
# Implement semantic search
def semantic_search(
    query,
    data_DF,
    embeddings,
    local_model = "all-MiniLM-L6-v2",
    k = 10,
    ):
    
    model = SentenceTransformer(local_model)
    query_embed = np.array(model.encode(query))
    batch_sims = batch_cos_sim(query_embed, embeddings)

    top_k = np.argsort(-batch_sims)[:k]
    results_DF = data_DF.iloc[top_k].copy()
    results_DF["cos_similarity"] = batch_sims[top_k]
    results_DF["result_rank"] = range(1, k + 1)

    return results_DF

In [29]:
# Test semantic search
chair_results = semantic_search(
    "chair",
    products_sample_DF,
    product_embeddings
    )

chair_results.info()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 199.07it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'pandas.DataFrame'>
Index: 10 entries, 6982 to 13008
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           10 non-null     int64  
 1   product_name         10 non-null     str    
 2   product_class        8 non-null      str    
 3   category_hierarchy   10 non-null     str    
 4   product_description  7 non-null      str    
 5   product_features     10 non-null     str    
 6   rating_count         6 non-null      float64
 7   average_rating       6 non-null      float64
 8   review_count         6 non-null      float64
 9   embed_text           10 non-null     str    
 10  cos_similarity       10 non-null     float32
 11  result_rank          10 non-null     int64  
dtypes: float32(1), float64(3), int64(2), str(6)
memory usage: 1000.0 bytes


In [30]:
slicer = ["result_rank", "cos_similarity", "product_name"]
chair_results[slicer]

,result_rank,cos_similarity,product_name
6982,1,0.743496,office chair
39109,2,0.691566,iredell task chair
26833,3,0.666003,ellender office chair
21610,4,0.655014,abberton task chair
18218,5,0.653960,ismene task chair
29621,6,0.648345,mike task chair
19468,7,0.645925,capucine task chair
5124,8,0.644453,aashritha task chair
15443,9,0.641115,tring office chair
13008,10,0.640363,sawin velvet task chair


_These look reasonable._

### 4b. Evaluate and compare BM25 vs semantic search (10 pts)

Implement Recall@k and compare the two search methods.

In [ ]:
# Implement Recall@k
def evaluate_search(
    search_func,
    queries_df: pd.DataFrame,
    labels_df: pd.DataFrame,
    k: int = 10,
    verbose: bool = True,
    ) -> pd.DataFrame:
    
    """
    Evaluate search across all queries using Recall@k.

    Recall@k = (# relevant items found in top k) / (total # relevant items)

    Args:
        search_func: Function that takes query string and returns DataFrame with product_id
        queries_df: DataFrame of queries
        labels_df: DataFrame with relevance labels
        k: Number of results to consider
        verbose: Whether to print progress

    Returns:
        DataFrame with query_id, query, and recall columns
    """
    
    results = []

    for _, row in queries_df.iterrows():
        query_id = row["query_id"]
        query_text = row["query"]

        search_results = search_func(query_text)
        product_ids = search_results["product_id"].tolist()[:k]

        # Recall calculation (grade > 0 = relevant)
        query_labels = labels_df[labels_df["query_id"] == query_id]
        relevant_ids = set(query_labels[query_labels["grade"] > 0]["product_id"])
        retrieved_ids = set(product_ids)
        recall = (
            len(retrieved_ids & relevant_ids) / len(relevant_ids)
            if relevant_ids
            else 0.0
        )

        results.append(
            {
                "query_id": query_id,
                "query": query_text,
                "recall": recall,
            }
        )

    results_df = pd.DataFrame(results)

    if verbose:
        print(f"Evaluated {len(results_df)} queries")
        # print(f"Mean Recall@{k}: {results_df['recall'].mean():.4f}")

    return results_df

In [ ]:
# Build BM25 index for comparison
sample_docs_list = list(products_sample_DF["embed_text"])
build_index(sample_docs_list, snowball_tokenize)

# Filter queries to those with products in our sample
sample_product_ids = set(products_sample_DF['product_id'])
sample_labels = labels[labels['product_id'].isin(sample_product_ids)]
sample_query_ids = set(sample_labels['query_id'])
sample_queries = queries[queries['query_id'].isin(sample_query_ids)]

print(f"Queries with products in sample: {len(sample_queries)}")

In [34]:
# Evaluate both BM25 and semantic search on all queries
# Calculate Recall@10 for each method

In [35]:
# Visualize comparison


---

## Task 5: Compare Embedding Models (20 points)

### 5a. Embed products with two different models (10 pts)

Compare embeddings from:
- `BAAI/bge-base-en-v1.5`
- `sentence-transformers/all-mpnet-base-v2`

In [36]:
# Load the two embedding models
bbe = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 211.08it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
amb = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 144.66it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [38]:

# Calculating embeddings through model.encode()
    # Ack. Prof. Matt Murphy's Quantitative Foundations course, Homework 1b
    #   for list-tqdm-vstack workflow
words_embeddings_aml6_list = []     # ml6 for all-MiniLM-L6-v2
for phrase in tqdm(words, desc = "Phrase embeds."):
    words_embeddings_aml6_list.append(
        model.encode(phrase)
        )

words_embeddings_aml6 = np.vstack(words_embeddings_aml6_list)

Phrase embeds.: 100%|██████████| 5/5 [00:00<00:00, 34.64it/s]


In [ ]:
# Embed products with both models
sample_embeddings_bbe_list = []
sample_embeddings_amb_list = []

for prod_nc in tqdm(products_sample_DF["embed_text"],
    desc = "Product embeds."):     # prod_nc for product name + product class
    sample_embeddings_bbe_list.append(
        bbe.encode(prod_nc)
        )

sample_embeddings_bbe = np.vstack(sample_embeddings_bbe_list)

for prod_nc in tqdm(products_sample_DF["embed_text"], desc = "Product embeds."):
    sample_embeddings_amb_list.append(
        amb.encode(prod_nc)
        )

sample_embeddings_amb = np.vstack(sample_embeddings_bbe_list)

Product embeds.: 100%|██████████| 5000/5000 [06:47<00:00, 12.28it/s]


In [ ]:
print(f"== ADDL. MODELS' EMBED. DIMENSIONS ==",     # ADDL. for additional
    f"\nBAAI/bge-base-en-v1.5: {sample_embeddings_bbe.shape[1]}",
    f"\nsentence-transformers/all-mpnet-base-v2: {sample_embeddings_amb.shape[1]}")

== ADDL. MODELS' EMBED. DIMENSIONS == 
BAAI/bge-base-en-v1.5: 768 
sentence-transformers/all-mpnet-base-v2: 768


### 5b. Compare search results between models (10 pts)

Evaluate both models on the same queries and analyze differences.

In [50]:
# Compare results for specific queries
test_queries = ["comfortable sofa", "star wars rug", "modern coffee table",
# add more!
    "bar cart", "green lampshade", "hexagonal tile", "pot rack"] #,
    # "knife block", "metal cutting board", "privacy curtains"]

for query in test_queries:
    print(f"== BGE RESULTS FOR QUERY \"{query}\" ==")
    bbe_results = semantic_search(
        query,
        products_sample_DF,
        sample_embeddings_bbe,
        local_model = "BAAI/bge-base-en-v1.5"
        )
    display(bbe_results)

    print(f"== MPNET RESULTS FOR QUERY \"{query}\" ==")
    amb_results = semantic_search(
        query,
        products_sample_DF,
        sample_embeddings_amb,
        local_model = "sentence-transformers/all-mpnet-base-v2"
        )
    display(amb_results)

== BGE RESULTS FOR QUERY "comfortable sofa" ==


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 175.22it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,product_id,product_name,product_class,category_hierarchy,product_description,product_features,rating_count,average_rating,review_count,embed_text,cos_similarity,result_rank
27405,27405,baeugris patio sofa with cushions,NaN,Outdoor / Outdoor & Patio Furniture / Outdoor Seating & Patio Chairs / Patio...,constructed of powder-coated steel frame make this wicker sectional sofa set...,seatheightwithoutcushion:14.2|producttype : patio sofa|dssecondaryproductsty...,NaN,NaN,NaN,baeugris patio sofa with cushions,0.808161,1
27406,27406,ashwini patio sofa with cushions,NaN,Outdoor / Outdoor & Patio Furniture / Outdoor Seating & Patio Chairs / Patio...,take an elegant approach when it comes to furnishing your backyard or patio ...,piecesincluded:4|overallproductweight:69|seatcushionthickness:1.96|seatingca...,NaN,NaN,NaN,ashwini patio sofa with cushions,0.802661,2
28691,28691,hamilton teak patio sofa with cushions,NaN,Outdoor / Outdoor & Patio Furniture / Outdoor Seating & Patio Chairs / Patio...,this sofa has a contemporary look using the finest teak . each piece of furn...,cushioncolor : canvas vellum|cushioncolor : canvas wheat|cushioncoverclosure...,NaN,NaN,NaN,hamilton teak patio sofa with cushions,0.800931,3
38543,38543,sofa bed with ottoman,NaN,Furniture / Living Room Furniture / Sofas,create a cozy spot in your living room with this sofa bed . the sofa bed and...,seatwidth-sidetoside:61|overallheight-toptobottom:36|seatfillmaterial : foam...,NaN,NaN,NaN,sofa bed with ottoman,0.798662,4
4040,4040,erasmus 75.6 '' velvet square arm sofa,NaN,Furniture / Living Room Furniture / Sofas,"this sleek , mid-century inspired velvet sofa is designed to impress . a lon...",upholsterycolor : blue|upholsterymaterial : velvet|levelofassembly : partial...,NaN,NaN,NaN,erasmus 75.6 '' velvet square arm sofa,0.797247,5
23200,23200,87 '' pillow top arm reclining sofa,Sofas,Furniture / Living Room Furniture / Sofas,"casual and comfortable , this reclining sofa is the perfect choice to update...",recliningtypedetails : manual - handle/lever|seatheight-floortoseat:20|armhe...,NaN,NaN,NaN,87 '' pillow top arm reclining sofa Sofas,0.794485,6
6209,6209,moss landing 81 '' flared arm sofa,Sofas,Furniture / Living Room Furniture / Sofas,NaN,upholsterycolor : beige performance ; 0863-93|legcolor : havana|legcolor : j...,5.0,5.0,5.0,moss landing 81 '' flared arm sofa Sofas,0.793009,7
9687,9687,kendall sectional sofa with ottoman,Sectionals,Furniture / Living Room Furniture / Sectionals,this sectional has a simple but elegant contemporary look that will combo we...,ottomandepth-fronttoback:23|upholsterycolor : gray|orientation : left hand f...,10.0,4.0,9.0,kendall sectional sofa with ottoman Sectionals,0.791797,8
23192,23192,eddison 90 '' rolled arm reclining sofa,Sofas,Furniture / Living Room Furniture / Sofas,this sofa is ready to be the centerpiece of your living room or home theater...,warrantylength : lifetime|seatwidth-sidetoside:69|pattern : solid color|leve...,1.0,5.0,1.0,eddison 90 '' rolled arm reclining sofa Sofas,0.791181,9
17180,17180,large bean bag sofa,NaN,Furniture / Game Tables & Game Room Furniture / Bean Bag Chairs,"not to be confused with a `` futon '' , this imperial fufton is a super cozy...",overalldepth-fronttoback:40|upholsterymaterial:100 % polyester|upholsterycol...,1056.0,4.0,812.0,large bean bag sofa,0.782461,10


== MPNET RESULTS FOR QUERY "comfortable sofa" ==


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 141.25it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

== MPNET RESULTS FOR QUERY "star wars rug" ==


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 151.19it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,product_id,product_name,product_class,category_hierarchy,product_description,product_features,rating_count,average_rating,review_count,embed_text,cos_similarity,result_rank
28503,28503,ghanaian rooster artisan crafted bird theme original mask wall décor,Wall Décor,Décor & Pillows / Wall Décor / Wall Accents,"a proud rooster basks in the west african sunshine , coming to life in a han...",shape : novelty|dssecondaryproductstyle : boho modern|dsprimaryproductstyle ...,1.0,4.0,1.0,ghanaian rooster artisan crafted bird theme original mask wall décor Wall Décor,0.125260,1
31405,31405,nydam dining table,Dining Tables,Furniture / Kitchen & Dining Furniture / Dining Tables & Seating / Kitchen &...,"with a scale appropriate for any number of smaller dining spaces , this nyda...","compatiblediningchairpartnumber : c007bk , c007gy , c007rd , c007wh|overallp...",174.0,4.5,109.0,nydam dining table Dining Tables,0.105013,2
42078,42078,5m 3528 smd rgb 300 led strip light string tape+44 key ir remote control,Under Cabinet Lighting,Lighting / Wall Lights / Under Cabinet Lighting,"these are 110 volt 50 ft multi-function led lights , suitable for indoor and...",rangeoffixture-minimumled : a|powersource : plug-in|overalldepth-fronttoback...,1.0,5.0,1.0,5m 3528 smd rgb 300 led strip light string tape+44 key ir remote control Und...,0.103021,3
28240,28240,gregoire coffee table,Coffee & Cocktail Tables,Furniture / Living Room Furniture / Coffee Tables & End Tables / Coffee Tables,one of a kind look with modern and industrial influences with this barrel ra...,dssecondaryproductstyle : transitional traditional|overallheight-toptobottom...,4.0,5.0,4.0,gregoire coffee table Coffee & Cocktail Tables,0.102666,4
2193,2193,5m 3528 rgb led strip strip strip strip lights smd lights string lights,Under Cabinet Lighting,Lighting / Wall Lights / Under Cabinet Lighting / Strip Under Cabinet Lighting,NaN,overallproductweight:0.24|whatisdrydamporwetlocationlisted : this indicates ...,NaN,NaN,NaN,5m 3528 rgb led strip strip strip strip lights smd lights string lights Unde...,0.101423,5
32052,32052,two roosters wall décor,Wall Décor,Décor & Pillows / Wall Décor / Wall Accents,NaN,countryoforigin : united states|overallwidth-sidetoside:11.5|countryoforigin...,1.0,3.0,1.0,two roosters wall décor Wall Décor,0.097995,6
32895,32895,knapp coffee table,Coffee & Cocktail Tables,Furniture / Living Room Furniture / Coffee Tables & End Tables / Coffee Tables,give your home a unique modern look with this beautiful coffee table . the c...,basecolor : black|warrantydetails:1 year limited guarantee|topcolor : beige|...,507.0,4.5,320.0,knapp coffee table Coffee & Cocktail Tables,0.097020,7
5906,5906,blonde dining table,Dining Tables,Furniture / Kitchen & Dining Furniture / Dining Tables & Seating / Kitchen &...,this contemporary dining table was inspired by the dining room of park avenu...,overallproductweight:83.7|overalldepth-fronttoback:31.5|topcolor : light gre...,9.0,4.5,8.0,blonde dining table Dining Tables,0.096132,8
17551,17551,lightbox 2,Table Lamps,Lighting / Table & Floor Lamps / Table Lamps / White Table Lamps,"they are a lamp , a seat , and an accent table all-in-one . the lamp ’ s com...",voltage:60|overallheight-toptobottom:17|cordcolor : black/white|countryofori...,4.0,5.0,4.0,lightbox 2 Table Lamps,0.095503,9
12204,12204,pineapple,NaN,Décor & Pillows / Art / All Wall Art,"great art deserves to be on canvas ! unlike thin posters and paper prints , ...",additionalmaterials : north american pine wood stretcher bars|worldcultures ...,NaN,NaN,NaN,pineapple,0.094443,10


== BGE RESULTS FOR QUERY "modern coffee table" ==


OSError: The paging file is too small for this operation to complete. (os error 1455)

In [ ]:
# Visualize model comparison with a scatter plot
# X-axis: BGE Recall@10, Y-axis: MPNet Recall@10

fig, ax = plt.subplots()

x = ["recall_10"]
y = ["recall_10"]

ax.scatter(x, y)

ax.set_xlabel("BGE Recall@10")
ax.set_ylabel("MPNet Recall@10")

plt.show()

---

## Task 6: Git Submission (5 points)

Submit your work using the Git workflow:

- [ ] Create a new branch called `homework-4`
- [ ] Commit your work with a meaningful message
- [ ] Push to GitHub
- [ ] Create a Pull Request
- [ ] Merge the PR to main
- [ ] Submit the `.ipynb` file on Blackboard

The TA will verify your submission by checking the merged PR on GitHub.